# Traveling Salesman Problem – Multiple Methods + Route Visualization

This notebook:
- generates random cities
- solves TSP with Nearest Neighbor, PuLP (MILP), Genetic Algorithm
- plots the actual path for each solution method
- compares total distances

In [ ]:
%matplotlib inline
import numpy as np
import pulp
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform

# ────────────────────────────────────────────────
#  Generate random cities
# ────────────────────────────────────────────────
np.random.seed(42)               # change or remove for different random instances
n_cities = 12
cities = np.random.rand(n_cities, 2) * 100
city_names = [f"C{i}" for i in range(n_cities)]

dist_matrix = squareform(pdist(cities, 'euclidean'))

print(f"Generated {n_cities} cities")
print("First 5 coordinates:\n", cities[:5])

In [ ]:
# ────────────────────────────────────────────────
#  Plotting helper – shows path with direction arrows
# ────────────────────────────────────────────────
def plot_tsp_route(cities, route, method_name, color='blue', figsize=(10,8)):
    fig, ax = plt.subplots(figsize=figsize)
    
    # Cities
    ax.scatter(cities[:,0], cities[:,1], s=220, c='white', edgecolors='navy', zorder=10)
    
    # Labels
    for i, (x,y) in enumerate(cities):
        ax.text(x+0.6, y+0.6, city_names[i], fontsize=10,
                ha='left', va='bottom', bbox=dict(facecolor='white', alpha=0.8))
    
    # Route line + markers
    r = cities[route]
    ax.plot(r[:,0], r[:,1], color=color, lw=3.2, marker='o', ms=10, zorder=5)
    
    # Arrows showing direction
    for i in range(len(route)-1):
        x1, y1 = r[i]
        x2, y2 = r[i+1]
        ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2.8,
                                    mutation_scale=22))
    
    # Start point
    ax.scatter(*cities[0], s=380, c='lime', marker='*',
               edgecolors='darkgreen', linewidth=2, zorder=15, label='Start')
    
    dist = sum(dist_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
    ax.set_title(f"{method_name}\nTotal distance: {dist:.1f}", fontsize=15)
    ax.grid(True, ls='--', alpha=0.35)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

## Nearest Neighbor (greedy heuristic)

In [ ]:
def nearest_neighbor(dist):
    n = len(dist)
    visited = [False] * n
    route = [0]
    visited[0] = True
    cur = 0
    total = 0
    for _ in range(n-1):
        nxt = np.argmin([dist[cur][j] if not visited[j] else np.inf for j in range(n)])
        route.append(nxt)
        visited[nxt] = True
        total += dist[cur][nxt]
        cur = nxt
    route.append(0)
    total += dist[cur][0]
    return route, total

nn_route, nn_dist = nearest_neighbor(dist_matrix)
print(f"Nearest Neighbor: {nn_dist:.1f}")

plot_tsp_route(cities, nn_route, "Nearest Neighbor", color="darkorange")

## PuLP MILP (exact) – using GLPK

In [ ]:
def solve_tsp_pulp(dist, n):
    prob = pulp.LpProblem("TSP", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("x", ((i,j) for i in range(n) for j in range(n) if i != j), cat="Binary")
    u = pulp.LpVariable.dicts("u", range(n), lowBound=1, upBound=n, cat="Integer")

    prob += pulp.lpSum(dist[i][j] * x[(i,j)] for i in range(n) for j in range(n) if i != j)

    for i in range(n):
        prob += pulp.lpSum(x[(i,j)] for j in range(n) if i != j) == 1
        prob += pulp.lpSum(x[(j,i)] for j in range(n) if i != j) == 1

    for i in range(1,n):
        for j in range(1,n):
            if i != j:
                prob += u[i] - u[j] + (n-1)*x[(i,j)] <= n-2

    # Use explicit path to GLPK solver binary
    glpk_path = "/opt/anaconda3/envs/opt/bin/glpsol"   # ← adjust this path if needed
    status = prob.solve(pulp.GLPK_CMD(path=glpk_path, msg=0))

    if pulp.LpStatus[status] != "Optimal":
        print("No optimal solution found. Status:", pulp.LpStatus[status])
        return None, None

    route = [0]
    cur = 0
    for _ in range(n-1):
        for j in range(n):
            if j != cur and (cur,j) in x and pulp.value(x[(cur,j)]) > 0.99:
                route.append(j)
                cur = j
                break
    route.append(0)
    return route, pulp.value(prob.objective)

pulp_route, pulp_dist = solve_tsp_pulp(dist_matrix, n_cities)

if pulp_dist is not None:
    print(f"PuLP optimal distance: {pulp_dist:.1f}")
    plot_tsp_route(cities, pulp_route, "PuLP MILP (exact)", color="purple")
else:
    print("PuLP failed to solve")

## Genetic Algorithm

In [ ]:
def route_dist(route, dist):
    return sum(dist[route[i]][route[(i+1)%len(route)]] for i in range(len(route)))

def genetic_algorithm(dist, pop_size=50, generations=150, mutation_rate=0.15):
    n = dist.shape[0]
    pop = [np.random.permutation(n).tolist() for _ in range(pop_size)]
    best_dist = float('inf')
    best_route = None

    for gen in range(generations):
        fitness = [route_dist(p + [p[0]], dist) for p in pop]
        best_idx = np.argmin(fitness)
        if fitness[best_idx] < best_dist:
            best_dist = fitness[best_idx]
            best_route = pop[best_idx][:]

        parents = sorted(range(pop_size), key=lambda i: fitness[i])[:pop_size//2]
        new_pop = [pop[i][:] for i in parents]

        while len(new_pop) < pop_size:
            p1, p2 = pop[np.random.choice(parents)], pop[np.random.choice(parents)]
            child = p1[:]
            start, end = sorted(np.random.randint(0, n, 2))
            child[start:end] = p1[start:end]
            remaining = [g for g in p2 if g not in child[start:end]]
            child = child[:start] + remaining[:start] + child[start:end] + remaining[start:]
            if np.random.rand() < mutation_rate:
                i, j = np.random.randint(0, n, 2)
                child[i], child[j] = child[j], child[i]
            new_pop.append(child)

        pop = new_pop[:pop_size]

    return best_route + [best_route[0]], best_dist

ga_route, ga_dist = genetic_algorithm(dist_matrix)
print(f"Genetic Algorithm best: {ga_dist:.1f}")

plot_tsp_route(cities, ga_route, "Genetic Algorithm", color="teal")

## Comparison of distances

In [ ]:
methods = []
dists = []

if 'nn_dist' in globals():
    methods.append('Nearest Neighbor')
    dists.append(nn_dist)

if 'pulp_dist' in globals() and pulp_dist is not None:
    methods.append('PuLP MILP')
    dists.append(pulp_dist)

if 'ga_dist' in globals():
    methods.append('Genetic Algorithm')
    dists.append(ga_dist)

if dists:
    plt.figure(figsize=(9,5))
    bars = plt.bar(methods, dists, color=['orange','purple','teal'])
    plt.ylabel('Total Distance')
    plt.title('TSP Solution Quality Comparison')
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f'{yval:.1f}', ha='center')
    plt.tight_layout()
    plt.show()
else:
    print("No solutions computed yet.")